# 00. Setup and your first decision

This notebook is the starting point. We will set up the environment, understand how a backend is configured, and make a first decision call.

A decision model is different from a chatbot. It does not write a sentence. It reads a state (the information it can see) and a set of questions whose possible answers are declared in advance. It then returns a probability for every declared answer. That small difference changes how we build systems: probabilities can be thresholded, compared, and audited.

Everything in this repository is free or open. We start with an offline mock so the notebook runs on any machine without keys, downloads, or a GPU.


## Step 1. Check the Python interpreter

The project pins Python 3.12 in the `.python-version` file. `uv` creates an isolated virtual environment in `.venv`, so nothing is installed into your system Python. The cell below only reports the interpreter version.


In [ ]:
import sys
print(sys.version.split()[0])


## Step 2. Import the project package

The package is installed in editable mode by `uv sync`. That means `decision_qa` points at the local `src/` folder: any edit to the source is visible immediately, with no reinstall. Printing the file path confirms we are importing our own code.


In [ ]:
import decision_qa
print(decision_qa.__file__)


## Step 3. Load the configuration

`load_config()` reads the `.env` file (if present) and returns a small dataclass. Two backends are supported: `local` for open models running on your machine, and `hosted` for a free-tier route. The default is `local`, so no secret is ever needed to follow the notebooks.


In [ ]:
from decision_qa import load_config
cfg = load_config()
print(cfg.backend)


The dataclass also holds the model name and the base URL. Let us read them separately so each cell demonstrates exactly one idea.


In [ ]:
print(cfg.model)
print(cfg.base_url)


## Step 4. Build the endpoint URL

The API we use is called `/v1/systemone`. Some base URLs already end in `/v1`, so the `endpoint` property removes the duplicate suffix. This is a small detail, but a wrong URL is the most common first error. The assertion verifies the result.


In [ ]:
print(cfg.endpoint)
assert cfg.endpoint.endswith('/v1/systemone')
print('endpoint OK')


## Step 5. Choose a client

A client is the object that sends a request and returns a result. `MockClient` is deterministic, free and offline. It is useful for learning the data structures and for testing code. `SystemOneClient` talks to a real server when one is available. We create both here, but we only call the mock for now.


In [ ]:
from decision_qa import MockClient
mock = MockClient(seed=42)
print(type(mock).__name__)


In [ ]:
from decision_qa import SystemOneClient
live = SystemOneClient(cfg)
print(type(live).__name__)


## Step 6. Define a state

The state is what the model is allowed to look at. In a question-answering task it usually contains a passage of text and a question. Keeping it as a dictionary makes it easy to extend later with JSON-like fields.


In [ ]:
state = {
    'context': 'A trial of 240 patients reports a 42 percent reduction in headaches.',
    'question': 'Is the treatment effective?',
}
print(state['question'])


In [ ]:
assert 'context' in state and 'question' in state
print('state keys OK')


## Step 7. The Noul question

Noul asks whether a statement is true. It is the simplest paradigm: the declared answers are `true` and `false`. We add a `gold` field for teaching and evaluation only. A real request would not include it.


In [ ]:
from decision_qa import NoulQuestion
noul = NoulQuestion(label='supported', statement='The treatment is effective.', gold=True)
print(noul.type, noul.label)


In [ ]:
from decision_qa import question_labels
print(question_labels(noul))
assert question_labels(noul) == ['true', 'false']


## Step 8. The Choice question

Choice asks the model to pick one of several declared options. The options are fixed by us, not generated by the model. The `gold_index` points at the correct option and is used only for scoring.


In [ ]:
from decision_qa import ChoiceQuestion
choice = ChoiceQuestion(label='verdict', options=['effective', 'ineffective', 'unclear'], gold_index=0)
print(choice.options)


In [ ]:
print(question_labels(choice))
assert 0 <= choice.gold_index < len(choice.options)
print('choice OK')


## Step 9. The Score question

Score uses an ordered rubric. Unlike Choice, the order carries meaning: `strong` is better than `moderate`. We should never shuffle the levels of a Score question.


In [ ]:
from decision_qa import ScoreQuestion
score = ScoreQuestion(label='evidence', rubric='Rate the strength of evidence.', levels=['weak', 'moderate', 'strong'], gold_index=2)
print(score.levels)


## Step 10. Build one request

The three question types are translated into a single payload that mirrors the `/v1/systemone` request shape. The `label` is not part of the wire format; it is our own handle for reading the results back.


In [ ]:
from decision_qa import to_systemone_payload
import json
payload = to_systemone_payload(state, [noul, choice, score])
print(json.dumps(payload, indent=2))


In [ ]:
assert len(payload['questions']) == 3
print('three questions in one payload')


## Step 11. Make the first call

We call the client with the state and a single Noul question. The result object holds the label of the question and a dictionary of probabilities. We inspect it one piece at a time.


In [ ]:
results = mock.decision(state, [noul])
print(results[0].question_label)


In [ ]:
probs = results[0].probabilities
print(probs)


In [ ]:
assert abs(sum(probs.values()) - 1.0) < 1e-9
print('probabilities sum to one')


## Step 12. Ask all three questions at once

Batching is one of the main advantages of this API shape. One state and several questions go together in a single call, which is cheaper and faster than three separate calls.


In [ ]:
all_results = mock.decision(state, [noul, choice, score])
print([r.question_label for r in all_results])


In [ ]:
assert len(all_results) == 3
print('one result per question')


In [ ]:
for r in all_results:
    print(r.question_label, max(r.probabilities, key=r.probabilities.get))


## Step 13. Change the state and observe the effect

The same question can receive a different distribution when the evidence changes. This is the core behaviour we will exploit later: decisions are conditioned on state.


In [ ]:
weak_state = {'context': 'A five-patient anecdote mentions improvement.', 'question': state['question']}
weak_results = mock.decision(weak_state, [noul])
print(weak_results[0].probabilities)


In [ ]:
strong = results[0].probabilities['true']
weak = weak_results[0].probabilities['true']
print(round(strong, 3), round(weak, 3))
assert strong > 0 and weak > 0


## Recap

We configured a backend, built the three question types, packed them into one payload, and read a probability distribution back. The next notebook examines each paradigm in more depth, including confidence thresholds and option order invariance.
